# Tutorial 99 — Live Demo: From Experimental Counts to Mutual Information

This notebook is intentionally more compact than the numbered feature tutorials, but it follows the same teaching pattern: first we explain what is happening, then the code shows the exact QCOM calls.

The demo uses the large plaintext count file in `example_data`, normalizes those counts into probabilities, prints the dominant bitstrings, and computes classical mutual information across a bipartition of the register.


## 1) Setup

Run this notebook from the `tutorials/` directory after installing the local package:

```bash
python -m pip install -e ".[dev,parquet,viz]"
```

The file path below is relative to `tutorials/`. QCOM's plaintext parser returns `(counts, total_count)`, where `counts` maps bitstrings to raw shot counts.


In [1]:
from pathlib import Path

import qcom as qc

file_path = Path("../example_data/1_billion_3.0_4_rungs.txt")
print(file_path)


../example_data/1_billion_3.0_4_rungs.txt


## 2) Load the count dictionary

The text reader expects lines of the form `<bitstring> <value>`. For experimental measurement data, the values are usually raw counts. We keep them as counts at first because count totals are meaningful: they tell us how many shots contributed to the empirical distribution.


In [2]:
counts, total_counts = qc.parse_text(file_path, show_progress=False)

print("unique bitstrings:", len(counts))
print("total counts:", int(total_counts))
qc.print_most_probable_bitstrings(counts, 10)


unique bitstrings: 105
total counts: 1000000000
Top 10 Most probable bit strings:
 1.  Bit string: 01000010, Probability: 389245564.00000000
 2.  Bit string: 10000001, Probability: 389223579.00000000
 3.  Bit string: 01000001, Probability: 48778624.00000000
 4.  Bit string: 10000010, Probability: 48777628.00000000
 5.  Bit string: 01000000, Probability: 20562997.00000000
 6.  Bit string: 00000001, Probability: 20561460.00000000
 7.  Bit string: 00000010, Probability: 20550353.00000000
 8.  Bit string: 10000000, Probability: 20549831.00000000
 9.  Bit string: 10000100, Probability: 4983517.00000000
10.  Bit string: 00100001, Probability: 4981922.00000000


## 3) Make the data type explicit

QCOM still accepts raw dictionaries for convenience, but typed containers remove ambiguity. `CountsData` validates that all keys are bitstrings of the same length and that the stored shot count matches the dictionary total.


In [3]:
counts_data = qc.CountsData({key: int(value) for key, value in counts.items()}, source=str(file_path))
print(counts_data)
print("shots:", counts_data.shots)
print("sites:", counts_data.n_sites)


CountsData(counts=mappingproxy({'11000011': 26175, '01010010': 240, '11001000': 5722, '10100010': 36, '11000010': 2216690, '11001010': 1, '11000000': 285826, '00010100': 2, '10110001': 1, '01000100': 72078, '00000000': 2211605, '10010000': 359220, '01100010': 541423, '11100010': 2, '01100000': 360363, '01001000': 4979884, '10000000': 20549831, '10000100': 4983517, '00110000': 4260, '00110001': 3776, '00010110': 7, '10100011': 1, '00000010': 20550353, '10011001': 512, '00100100': 14528, '10100000': 13, '10001100': 3933, '10010001': 541962, '01100101': 2, '00101001': 7, '11100001': 6, '01000010': 389245564, '10000111': 5, '01000101': 27, '00100011': 5603, '10000110': 1075458, '01001011': 1, '00001000': 462524, '01000000': 20562997, '01000001': 48778624, '01000011': 2218892, '01110001': 1, '00010010': 4981597, '01101000': 1, '00100010': 71868, '11000001': 2213693, '01000110': 540677, '11010010': 2, '00011000': 14329, '10010010': 1075772, '01011000': 4, '00001100': 4256, '10000011': 221530

## 4) Normalize counts to probabilities

Entropy and mutual information are probability-distribution quantities. We therefore normalize the count dictionary before calling information metrics. The dictionary-returning path is shown first because it is lightweight and familiar; the container-returning path is useful when you want validated metadata to travel with the probabilities.


In [4]:
probabilities = qc.normalize_to_probabilities(counts_data)
probability_data = qc.normalize_to_probabilities(counts_data, return_data=True)

print("probability sum:", sum(probabilities.values()))
print("container:", probability_data)
qc.print_most_probable_bitstrings(probability_data.to_dict(), 10)


probability sum: 1.0
container: ProbabilityData(probabilities=mappingproxy({'11000011': 2.6175e-05, '01010010': 2.4e-07, '11001000': 5.722e-06, '10100010': 3.6e-08, '11000010': 0.00221669, '11001010': 1e-09, '11000000': 0.000285826, '00010100': 2e-09, '10110001': 1e-09, '01000100': 7.2078e-05, '00000000': 0.002211605, '10010000': 0.00035922, '01100010': 0.000541423, '11100010': 2e-09, '01100000': 0.000360363, '01001000': 0.004979884, '10000000': 0.020549831, '10000100': 0.004983517, '00110000': 4.26e-06, '00110001': 3.776e-06, '00010110': 7e-09, '10100011': 1e-09, '00000010': 0.020550353, '10011001': 5.12e-07, '00100100': 1.4528e-05, '10100000': 1.3e-08, '10001100': 3.933e-06, '10010001': 0.000541962, '01100101': 2e-09, '00101001': 7e-09, '11100001': 6e-09, '01000010': 0.389245564, '10000111': 5e-09, '01000101': 2.7e-08, '00100011': 5.603e-06, '10000110': 0.001075458, '01001011': 1e-09, '00001000': 0.000462524, '01000000': 0.020562997, '01000001': 0.048778624, '01000011': 0.002218892, 

## 5) Choose a bipartition

The bitstrings in this dataset have eight sites. Here we split the first four sites into region A and the last four sites into region B. QCOM uses a `configuration` list to mark regions: `0` for A and `1` for B in the classical mutual-information helpers.


In [5]:
configuration = [0] * 4 + [1] * 4
print("configuration:", configuration)
print("bitstring length:", probability_data.n_sites)


configuration: [0, 0, 0, 0, 1, 1, 1, 1]
bitstring length: 8


## 6) Compute mutual information

The scalar default is convenient when you only need $I(A;B)$. Passing `return_components=True` returns `MutualInformationResult`, which also exposes the three entropies used internally:

$$
I(A;B) = H(A) + H(B) - H(AB).
$$


In [6]:
mutual_information = qc.compute_mutual_information(
    probabilities,
    configuration=configuration,
    base=2,
)
components = qc.compute_mutual_information(
    probabilities,
    configuration=configuration,
    base=2,
    return_components=True,
)

print("I(A;B) bits:", mutual_information)
print("H(A):", components.h_a)
print("H(B):", components.h_b)
print("H(AB):", components.h_ab)
print("dataclass type:", type(components).__name__)


I(A;B) bits: 0.4553241887935213
H(A): 1.3833866626952411
H(B): 1.3834229944052456
H(AB): 2.3114854683069654
dataclass type: MutualInformationResult


## 7) What this demo showed

In a few QCOM calls we moved from a raw experimental-style count file to a normalized probability distribution and then to an information-theoretic summary. The important habit is to keep counts and probabilities conceptually separate: counts carry shot statistics, while probabilities are the right input for entropy and mutual-information calculations.
